# 18_4_3 DEM Sizes

This notebook extracts DEM-derived decoding matrix sizes for the bicycle-bivariate `18_4_3` test circuits.

- `XZ` means basis-filtered decoding, matching `xyz=False` in the older notebooks.
- `XYZ` means full unfiltered decoding, matching `xyz=True` in the older notebooks.

In [1]:
from pathlib import Path

import pandas as pd
import stim

from relay_bp.stim.sinter.check_matrices import CheckMatrices
from relay_bp.stim.sinter.runner import _filter_detectors_by_basis as filter_detectors_by_basis


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "tests" / "testdata" / "bicycle_bivariate").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find repo root containing tests/testdata/bicycle_bivariate "
        f"starting from {start}."
    )


repo_root = find_repo_root(Path.cwd().resolve())
circuits_dir = repo_root / "tests" / "testdata" / "bicycle_bivariate"

print("repo_root =", repo_root)
print("circuits_dir =", circuits_dir)


repo_root = C:\Users\User\Documents\projects-git\relay_mother_folder\relay
circuits_dir = C:\Users\User\Documents\projects-git\relay_mother_folder\relay\tests\testdata\bicycle_bivariate


In [2]:
def parse_name_metadata(path: Path) -> dict:
    metadata = {"stim_path": str(path)}
    for part in path.stem.split(","):
        if "=" not in part:
            continue
        key, value = part.split("=", 1)
        metadata[key] = value
    if "error_rate" in metadata:
        metadata["p"] = float(metadata["error_rate"])
    return metadata


def collect_dem_sizes() -> pd.DataFrame:
    rows = []
    for path in sorted(circuits_dir.glob("circuit=bicycle_bivariate_18_4_3_memory_*.stim")):
        if "memory_choi_XZ" in path.name:
            continue

        metadata = parse_name_metadata(path)
        circuit_name = metadata["circuit"]
        basis = circuit_name[-1]

        circuit_xyz = stim.Circuit.from_file(path)
        dem_xyz = circuit_xyz.detector_error_model()
        matrices_xyz = CheckMatrices.from_dem(dem_xyz, prune_decided_errors=False)

        circuit_xz = filter_detectors_by_basis(circuit_xyz, basis)
        dem_xz = circuit_xz.detector_error_model()
        matrices_xz = CheckMatrices.from_dem(dem_xz, prune_decided_errors=False)

        rows.append(
            {
                "circuit": circuit_name,
                "basis": basis,
                "p": metadata["p"],
                "xz_rows": matrices_xz.check_matrix.shape[0],
                "xz_cols": matrices_xz.check_matrix.shape[1],
                "xyz_rows": matrices_xyz.check_matrix.shape[0],
                "xyz_cols": matrices_xyz.check_matrix.shape[1],
                "xz_check_nnz": int(matrices_xz.check_matrix.nnz),
                "xyz_check_nnz": int(matrices_xyz.check_matrix.nnz),
                "xz_obs_rows": matrices_xz.observables_matrix.shape[0],
                "xz_obs_cols": matrices_xz.observables_matrix.shape[1],
                "xyz_obs_rows": matrices_xyz.observables_matrix.shape[0],
                "xyz_obs_cols": matrices_xyz.observables_matrix.shape[1],
                "xz_detectors": dem_xz.num_detectors,
                "xyz_detectors": dem_xyz.num_detectors,
                "xz_observables": dem_xz.num_observables,
                "xyz_observables": dem_xyz.num_observables,
                "stim_path": str(path),
            }
        )

    if not rows:
        raise FileNotFoundError(
            f"No matching 18_4_3 circuits found in {circuits_dir}."
        )

    return pd.DataFrame(rows).sort_values(["circuit", "p"]).reset_index(drop=True)


sizes_df = collect_dem_sizes()
display(
    sizes_df[
        [
            "circuit",
            "p",
            "xz_rows",
            "xz_cols",
            "xyz_rows",
            "xyz_cols",
            "xz_check_nnz",
            "xyz_check_nnz",
        ]
    ]
)


,circuit,p,xz_rows,xz_cols,xyz_rows,xyz_cols,xz_check_nnz,xyz_check_nnz
0,bicycle_bivariate_18_4_3_memory_X,0.0010,36,288,54,1818,891,8874
1,bicycle_bivariate_18_4_3_memory_X,0.0015,36,288,54,1818,891,8874
2,bicycle_bivariate_18_4_3_memory_X,0.0020,36,288,54,1818,891,8874
3,bicycle_bivariate_18_4_3_memory_X,0.0025,36,288,54,1818,891,8874
4,bicycle_bivariate_18_4_3_memory_X,0.0030,36,288,54,1818,891,8874
5,bicycle_bivariate_18_4_3_memory_X,0.0035,36,288,54,1818,891,8874
6,bicycle_bivariate_18_4_3_memory_X,0.0040,36,288,54,1818,891,8874
7,bicycle_bivariate_18_4_3_memory_X,0.0045,36,288,54,1818,891,8874
8,bicycle_bivariate_18_4_3_memory_X,0.0050,36,288,54,1818,891,8874
9,bicycle_bivariate_18_4_3_memory_X,0.0060,36,288,54,1818,891,8874


In [3]:
summary_df = (
    sizes_df.groupby(["circuit", "basis"], as_index=False)
    .agg(
        p_min=("p", "min"),
        p_max=("p", "max"),
        num_files=("p", "size"),
        xz_rows=("xz_rows", "first"),
        xz_cols=("xz_cols", "first"),
        xyz_rows=("xyz_rows", "first"),
        xyz_cols=("xyz_cols", "first"),
        xz_check_nnz=("xz_check_nnz", "first"),
        xyz_check_nnz=("xyz_check_nnz", "first"),
    )
)

summary_df["xz_check_shape"] = summary_df.apply(
    lambda row: f"{row['xz_rows']} x {row['xz_cols']}", axis=1
)
summary_df["xyz_check_shape"] = summary_df.apply(
    lambda row: f"{row['xyz_rows']} x {row['xyz_cols']}", axis=1
)

display(
    summary_df[
        [
            "circuit",
            "basis",
            "p_min",
            "p_max",
            "num_files",
            "xz_check_shape",
            "xyz_check_shape",
            "xz_check_nnz",
            "xyz_check_nnz",
        ]
    ]
)


,circuit,basis,p_min,p_max,num_files,xz_check_shape,xyz_check_shape,xz_check_nnz,xyz_check_nnz
0,bicycle_bivariate_18_4_3_memory_X,X,0.001,0.01,14,36 x 288,54 x 1818,891,8874
1,bicycle_bivariate_18_4_3_memory_Z,Z,0.001,0.01,14,36 x 288,54 x 1800,945,8910


In [4]:
shape_cols = [
    "xz_rows",
    "xz_cols",
    "xyz_rows",
    "xyz_cols",
    "xz_check_nnz",
    "xyz_check_nnz",
]

consistency_df = sizes_df.groupby("circuit")[shape_cols].nunique()
display(consistency_df)

if not (consistency_df == 1).all().all():
    raise ValueError("DEM sizes are not constant across p for at least one circuit.")

print("DEM sizes are constant across p for each 18_4_3 circuit family.")


,xz_rows,xz_cols,xyz_rows,xyz_cols,xz_check_nnz,xyz_check_nnz
circuit,,,,,,
bicycle_bivariate_18_4_3_memory_X,1,1,1,1,1,1
bicycle_bivariate_18_4_3_memory_Z,1,1,1,1,1,1


DEM sizes are constant across p for each 18_4_3 circuit family.
